**Note:** this notebook downloads the screened on-site meteorological data from the internal Swiss FluxNet database and can only be run within ETH Zurich.

# **Settings**

## Variables

In [ ]:
MEASUREMENTS = ['TA', 'RH', 'PA', 'PPFD', 'SW', 'LW', 'G','SWC', 'TS', 'PREC']  # Measurement name
FIELDS = ['PREC_M1_2_1',
          'TA_M1_2_1', 
          'RH_M1_2_1', 
          'PA_M1_2_1', 
          'PPFD_IN_T1_2_1',
          'SW_BC_IN_T1_2_1', 'SW_IN_T1_2_1', 'SW_BC_OUT_T1_2_1', 'SW_OUT_T1_2_1',
          'LW_BC_IN_T1_2_1', 'LW_IN_T1_2_1', 'LW_BC_OUT_T1_2_1', 'LW_OUT_T1_2_1',
          'SWC_CF2_0.05_1', 'SWC_CF2_0.15_1', 'SWC_CF2_0.3_1',
          'SWC_CF3_0.05_3', 'SWC_CF3_0.1_3', 'SWC_CF3_0.2_3', 'SWC_CF3_0.3_3', 'SWC_CF3_0.4_3', 'SWC_CF3_0.5_3', # SOIL VUE
          'TS_CF2_0.05_1', 'TS_CF2_0.15_1', 'TS_CF2_0.3_1',
          'TS_CF3_0.05_3', 'TS_CF3_0.1_3', 'TS_CF3_0.2_3', 'TS_CF3_0.3_3', 'TS_CF3_0.4_3', 'TS_CF3_0.5_3', # SOIL VUE
          'G_CF1_0.05_1'
          ]

## Database settings

In [ ]:
# Settings for database download
SITE = 'ch-tan'  # Site name
# Start and stop timestamps need to be specified as strings in the format 'YYYY-MM-DD HH:MM:SS' for diive database download. The timestamps are in UTC timezone, so they need to be converted from the local timezone (CET, UTC+1) to UTC.
START = '2023-10-17 00:00:00'                                     # '2023-10-17 00:00:00'
STOP = '2025-06-06 00:00:00' 
DATA_VERSION = 'meteoscreening_diive'
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # Timezone, e.g. "1" is translated to timezone "UTC+01:00" (CET, winter time)
data_version = "meteoscreening_diive"
DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'  # Folder with configurations

</br>

# **Imports**

In [ ]:
import importlib.metadata
import warnings
from datetime import datetime

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np

from dbc_influxdb import dbcInflux

from diive.core.plotting.heatmap_datetime import HeatmapDateTime  # For plotting heatmaps
from diive.core.plotting.timeseries import TimeSeries  # For simple (interactive) time series plotting
from diive.pkgs.formats.meteo import FormatMeteoForFluxnetUpload  # Class to format output files for upload

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)

version_diive = importlib.metadata.version("diive")
print(f"diive version: v{version_diive}")

</br>

# **Docstring**

In [ ]:
# help(FormatMeteoForEddyProProcessing)

</br>

# **Load example data**
- This example uses data from a database.

In [ ]:
dbc = dbcInflux(dirconf=DIRCONF)
df, _, _ = \
    dbc.download(bucket=f'{SITE}_processed',
                 measurements=MEASUREMENTS,
                 fields=FIELDS,
                 start=START,
                 stop=STOP,
                 timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
                 data_version=DATA_VERSION)

The dataframe with original data looks like this:

In [ ]:
df

</br>

# **Rename variables**

Merge variables with duplicated naming

In [ ]:
df['sw_in'] = df['SW_BC_IN_T1_2_1'].combine_first(df['SW_IN_T1_2_1'])
df['sw_out'] = df['SW_BC_OUT_T1_2_1'].combine_first(df['SW_OUT_T1_2_1'])
df['lw_in'] = df['LW_BC_IN_T1_2_1'].combine_first(df['LW_IN_T1_2_1'])
df['lw_out'] = df['LW_BC_OUT_T1_2_1'].combine_first(df['LW_OUT_T1_2_1'])

df.drop(columns=['SW_BC_IN_T1_2_1', 'SW_IN_T1_2_1', 'SW_BC_OUT_T1_2_1', 'SW_OUT_T1_2_1', 'LW_BC_IN_T1_2_1', 'LW_IN_T1_2_1', 'LW_BC_OUT_T1_2_1', 'LW_OUT_T1_2_1'], inplace=True)

df

In [ ]:
# Rename original variables
rename_dict = {
    'TA_M1_2_1': 'ta',
    'RH_M1_2_1': 'rh',
    'PA_M1_2_1': 'pa',
    'PREC_M1_2_1': 'prec',
    'PPFD_IN_T1_2_1': 'ppfd', 
    'SW_OUT_T1_2_1': 'sw_out',
    'LW_IN_T1_2_1': 'lw_in', 
    'LW_OUT_T1_2_1': 'lw_out', 
    'SWC_CF2_0.05_1': 'swc_0.05',
    'SWC_CF2_0.15_1': 'swc_0.15',
    'SWC_CF2_0.3_1': 'swc_0.3',
    'SWC_CF3_0.05_3': 'swc_0.05_soilvue', 
    'SWC_CF3_0.1_3': 'swc_0.1_soilvue',
    'SWC_CF3_0.2_3': 'swc_0.2_soilvue',
    'SWC_CF3_0.3_3': 'swc_0.3_soilvue',
    'SWC_CF3_0.4_3': 'swc_0.4_soilvue',
    'SWC_CF3_0.5_3': 'swc_0.5_soilvue',
    'TS_CF2_0.05_1': 'ts_0.05', 
    'TS_CF2_0.15_1': 'ts_0.15', 
    'TS_CF2_0.3_1': 'ts_0.3',
    'TS_CF3_0.05_3': 'ts_0.05_soilvue', 
    'TS_CF3_0.1_3': 'ts_0.1_soilvue',
    'TS_CF3_0.2_3': 'ts_0.2_soilvue',
    'TS_CF3_0.3_3': 'ts_0.3_soilvue',
    'TS_CF3_0.4_3': 'ts_0.4_soilvue',
    'TS_CF3_0.5_3': 'ts_0.5_soilvue', 
    'G_CF1_0.05_1': 'g'
}

df.rename(columns=rename_dict, inplace=True)

</br>

# **Plot all variables**

In [ ]:
plotcols = [plotcol for plotcol in df.columns if not "TIMESTAMP" in plotcol]
plotdf = df[plotcols].copy()

for col in plotdf.columns:
    series = plotdf[col].copy()
    try:
        fig = plt.figure(facecolor='white', figsize=(4, 8))
        gs = gridspec.GridSpec(1, 1)  # rows, cols
        ax = fig.add_subplot(gs[0, 0])
        vmin = series.quantile(.01)
        vmax = series.quantile(.99)
        HeatmapDateTime(ax=ax, series=series, vmin=vmin, vmax=vmax).plot()
        ax.set_title(col, color='black')
        fig.show()
    except TypeError:
        pass

</br>

# **Save to CSV**

In [ ]:
df.to_csv('11.1_CH-TAN_meteo_meteoscreening_diive.csv')

</br>

# **End of notebook**

In [ ]:
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Finished {dt_string}")